In [2]:
%pip install selenium
%pip install numpy
%pip install yfinance
%pip install pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import json
import time

HEADERS = {
    "User-Agent": "Sathvik Malla sathvikmalla17@gmail.com",
    "Accept-Encoding": "gzip, deflate",
}

BASE = "https://data.sec.gov"

In [5]:
def get_cik(ticker):
    url = "https://efts.sec.gov/LATEST/search-index?q=%22{}%22&dateRange=custom&startdt=2020-01-01&forms=10-K".format(ticker)

    tickers_url = "https://www.sec.gov/files/company_tickers.json"
    r = requests.get(tickers_url, headers=HEADERS)
    data = r.json()

    for entry in data.values():
        if entry["ticker"].upper() == ticker.upper():
            return str(entry["cik_str"]).zfill(10)
    
    raise ValueError(f"Ticker {ticker} not found")

cik = get_cik("NVDA")
print(cik)

0001045810


In [6]:
def get_filings(cik, form_type="10-K", count=5):
    url = f"{BASE}/submissions/CIK{cik}.json"
    r = requests.get(url, headers=HEADERS)
    data = r.json()

    filings = data["filings"]["recent"]
    results = []

    for i, form in enumerate(filings["form"]):
        if form == form_type:
            results.append({
                "accessionNumber": filings["accessionNumber"][i].replace("-", ""),
                "filingDate": filings["filingDate"][i],
                "form": form,
                "primaryDocument": filings["primaryDocument"][i]
            })
            if len(results) >= count:
                break
    
    return results

In [7]:
SECTION_TARGETS = {
    "supply_chain":  ["supply chain", "supplier", "suppliers"],
    "vendors":       ["vendors", "vendor relationships", "vendor concentration"],
    "manufacturing": ["manufacturing partners", "contract manufacturer", "outsourced manufacturing"],
    "raw_materials": ["raw materials", "components", "procurement"],
}

def extract_section(text, section_keywords, next_section_keywords = None):
    text_lower = text.lower()
    start_idx = -1
    
    for keyword in section_keywords:
        idx = text_lower.find(keyword.lower())
        if idx != -1:
            start_idx = idx
            break
    
    if start_idx == -1:
        return ""
    
    end_idx = len(text)
    if next_section_keywords:
        for keyword in next_section_keywords:
            idx = text_lower.find(keyword.lower(), start_idx + 100)
            if idx != -1:
                end_idx = min(end_idx, idx)
    else:
        end_idx = start_idx + 5000
    
    return text[start_idx:end_idx].strip()

In [8]:
def get_filing_text(cik, accession, doc_name):
    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{doc_name}"
    r = requests.get(url, headers=HEADERS)
    time.sleep(0.5)  # Respect rate limits
    return r.text

def extract_relationships_from_10k(text, company):
    """
    10K: partnerships, suppliers, customers, subsidiaries, JVs.
    """
    import re
    
    relationships = []
    
    patterns = {
        "partnership": r"(?:partnership|partnered|strategic alliance).*?(?:with|between)\s+([A-Z][A-Za-z\s,]+(?:Inc|Corp|LLC|Ltd|Co\.)?)",
        "supplier": r"([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)\s+(?:supplies|is (?:a|our) (?:key |primary )?supplier)",
        "customer": r"([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)\s+(?:is|was|are|were)\s+(?:a|our|the)\s+(?:significant|major|key|largest)\s+customer",
        "acquisition": r"(?:acquired|acquisition of|merger with)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "investment": r"(?:invested in|investment in|equity stake in)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "subsidiary": r"(?:wholly.owned subsidiary|our subsidiary)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        
        "integration":    r"(?:integrates?|integration with|native(?:ly)? integrat)\s+(?:with\s+)?([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "reseller":       r"(?:reseller|resell|channel partner|VAR)\s+(?:agreement with\s+)?([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "cloud_platform": r"(?:built on|deployed on|runs on|hosted on)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)\s+(?:cloud|platform|infrastructure)",
        "data_sharing":   r"(?:data sharing|data exchange|API access)\s+(?:agreement with\s+)?([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "competitor":     r"(?:compete(?:s)? with|competition from|competitive(?:ly)? with)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
    }

    supply_chain_patterns = {
        "supplier":              r"([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)\s+(?:supplies|is (?:a|our) (?:key |primary |sole )?supplier)",
        "sole_supplier":         r"sole\s+(?:source|supplier)\s+(?:of|for)?\s*([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "vendor":                r"(?:vendor|vendors)\s+(?:including|such as|like)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "manufacturing_partner": r"(?:manufactured by|manufacturing partner|contract manufacturer)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "raw_material_source":   r"(?:source|sourced|procured)\s+(?:from)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "vendor_concentration":  r"(?:concentration|dependent|dependency)\s+(?:on|upon)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
    }
    
    for rel_type, pattern in patterns.items():
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches[:10]:
            relationships.append({
                "source": company,
                "target": match.strip(),
                "type": rel_type,
                "filing": "10-K"
            })
    
    sections = {
        "supply_chain": extract_section(
            text,
            section_keywords=SECTION_TARGETS["supply_chain"],
            next_section_keywords=["item 1a", "risk factors", "item 2", "vendors"]
        ),
        "vendors": extract_section(
            text,
            section_keywords=SECTION_TARGETS["vendors"],
            next_section_keywords=["item 1a", "risk factors", "item 2", "manufacturing"]
        ),
        "manufacturing": extract_section(
            text,
            section_keywords=SECTION_TARGETS["manufacturing"],
            next_section_keywords=["item 1a", "risk factors", "item 2", "raw materials"]
        ),
        "raw_materials": extract_section(
            text,
            section_keywords=SECTION_TARGETS["raw_materials"],
            next_section_keywords=["item 1a", "risk factors", "item 2"]
        ),
    }

    for section_name, section_text in sections.items():
        if not section_text:
            continue
        
        for rel_type, pattern in supply_chain_patterns.items():
            matches = re.findall(pattern, section_text, re.IGNORECASE)
            for match in matches[:5]:
                relationships.append({
                    "source":  company,
                    "target":  match.strip(),
                    "type":    rel_type,
                    "section": section_name,
                    "filing":  "10-K",
                })
    
    return relationships

In [9]:
def get_ownership_filings(cik: str) -> list:
    results = []
    
    for form in ["SC 13D", "SC 13G"]:
        filings = get_filings(cik, form_type=form, count=10)
        for f in filings:
            results.append({
                "form": form,
                "date": f["filingDate"],
                "accession": f["accessionNumber"],
                "influence_type": "activist" if "13D" in form else "passive",
                "target_company_cik": cik,
            })
    
    return results

def parse_13d_for_investor(cik: str) -> list[dict]:
    filings = get_filings(cik, form_type="SC 13D", count=3)
    ownership_data = []
    
    for filing in filings:
        url = f"{BASE}/submissions/CIK{cik}.json"
        # Parse filer name, ownership %, purpose of transaction
        # from the filing document text
        ownership_data.append({
            "filing_date": filing["filingDate"],
            "accession": filing["accessionNumber"],
        })
    
    return ownership_data

In [10]:
import yfinance as yf
import pandas as pd

def get_yfinance_relationships(ticker: str) -> dict:
    stock = yf.Ticker(ticker)
    
    def safe_records(attr) -> list:
        val = getattr(stock, attr, None)
        if val is None:
            return []
        if isinstance(val, pd.DataFrame):
            return val.head(5).to_dict("records")
        if isinstance(val, dict):
            return [val]  # wrap bare dict in a list
        return []

    return {
        "ticker": ticker,
        "info": {
            "sector": stock.info.get("sector"),
            "industry": stock.info.get("industry"),
            "fullTimeEmployees": stock.info.get("fullTimeEmployees"),
            "longBusinessSummary": stock.info.get("longBusinessSummary", "")[:500],
        },
        "institutional_holders": safe_records("institutional_holders"),
        "major_holders": safe_records("major_holders"),
    }
       

def extract_shared_holders(tickers: list) -> list:
    """
    Find institutional investors that hold MULTIPLE companies —
    these cross-holdings imply relational connections.
    """
    holder_map = {}  # institution -> [tickers]
    
    for ticker in tickers:
        data = get_yfinance_relationships(ticker)
        for holder in data["institutional_holders"]:
            name = holder.get("Holder", "")
            if name:
                holder_map.setdefault(name, []).append(ticker)
    
    # Return only cross-holders (hold 2+ companies)
    return [
        {"institution": inst, "holds": tickers_held}
        for inst, tickers_held in holder_map.items()
        if len(tickers_held) > 1
    ]

In [11]:
%pip install selenium webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [12]:
from selenium.webdriver.chrome.webdriver import WebDriver as ChromeDriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

def create_driver(headless=True):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--user-agent=Mozilla/5.0 (Macintosh; ARM Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

    
    driver = ChromeDriver(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver

In [13]:
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

def scrape_bloomberg_company(ticker: str) -> dict:
    driver = create_driver()
    relationships = []
    
    try:
        url = f"https://www.bloomberg.com/quote/{ticker}:US"
        driver.get(url)
        time.sleep(3)  # Wait for JS render
        
        wait = WebDriverWait(driver, 10)
        
        # Extract company description
        try:
            desc = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "[data-component='description']"))
            ).text
        except:
            desc = ""
        
        # Extract peer companies (shown publicly)
        try:
            peers = driver.find_elements(By.CSS_SELECTOR, "[data-component='peer-table'] tr")
            for peer in peers:
                ticker_el = peer.find_elements(By.TAG_NAME, "td")
                if ticker_el:
                    relationships.append({
                        "type": "industry_peer",
                        "target": ticker_el[0].text,
                    })
        except:
            pass
        
        return {"ticker": ticker, "description": desc, "peers": relationships}
    
    finally:
        driver.quit()

In [14]:
def scrape_berkshire_portfolio() -> list:
    driver = create_driver()
    holdings = []
    
    try:
        driver.get("https://www.berkshirehathaway.com/invest.html")
        time.sleep(2)
        
        links = driver.find_elements(By.TAG_NAME, "a")
        for link in links:
            text = link.text.strip()
            href = link.get_attribute("href") or ""
            if text and not text.startswith("http"):
                holdings.append({"company": text, "url": href})
        
        return holdings
    
    finally:
        driver.quit()

def scrape_berkshire_annual_letter(year: int = 2024) -> str:
    driver = create_driver()
    
    try:
        # Letters are PDFs on their site
        url = f"https://www.berkshirehathaway.com/letters/{year}ltr.pdf"
        driver.get(url)
        time.sleep(3)
        return driver.find_element(By.TAG_NAME, "body").text
    finally:
        driver.quit()

In [15]:
from urllib.parse import quote

def search_edgar_for_relationship(company_a: str, company_b: str) -> list:

    """
    Search EDGAR full-text search for filings mentioning BOTH companies.
    Useful for finding cross-company references in 10-Ks.
    """
    driver = create_driver()
    results = []
    
    try:
        query = f'"{company_a}" "{company_b}"'
        url = f"https://efts.sec.gov/LATEST/search-index?q={quote(query)}&dateRange=custom&startdt=2024-01-01&forms=10-K,8-K"
        
        # Use the API directly (faster than Selenium for EDGAR)
        api_url = f"https://efts.sec.gov/LATEST/search-index?q={quote(query)}&forms=10-K"
        r = requests.get(api_url, headers=HEADERS)
        data = r.json()
        
        for hit in data.get("hits", {}).get("hits", [])[:5]:
            results.append({
                "company": hit["_source"].get("entity_name"),
                "filing_date": hit["_source"].get("file_date"),
                "form": hit["_source"].get("form_type"),
                "accession": hit["_source"].get("accession_no"),
            })
        
        return results
    finally:
        driver.quit()

In [ ]:
import json
from datetime import datetime

INVESTMENT_TRACKS = {
    "CLOUD_ENTERPRISE_SOFTWARE": ["CRM", "ORCL", "NOW", "ADBE", "WDAY"],
}

def build_relationship_graph(tickers: list, track_name: str) -> dict:
    """
    Master function: pulls from all sources and builds
    the node/link graph for a given investment track.
    """
    nodes = []
    links = []
    sources_used = []
    
    print(f"\n📊 Building graph for track: {track_name}")
    print(f"   Tickers: {', '.join(tickers)}\n")
    
    # --- Step 1: Build nodes from yFinance ---
    for ticker in tickers:
        print(f"  → yFinance: {ticker}")
        try:
            yf_data = get_yfinance_relationships(ticker)
            sections = {}
            try:
                cik = get_cik(ticker)
                filings = get_filings(cik, "10-K", count=1)
                if filings:
                    f = filings[0]
                    raw_text = get_filing_text(cik, f["accessionNumber"], f["primaryDocument"])
                    time.sleep(1)
                    
                    sections = {
                        name: extract_section(
                            raw_text,
                            config["start"],
                            config["end"]
                        )
                        for name, config in SECTION_TARGETS.items()
                    }
                    # Drop empty sections before storing
                    sections = {k: v for k, v in sections.items() if v.strip()}
                    print(f"    ✅ Sections found: {list(sections.keys())}")
            except Exception as e:
                print(f"    ⚠ Could not fetch 10-K sections for {ticker}: {e}")
            
            nodes.append({
                "id": ticker,
                "name": yf_data["info"].get("longName", ticker),
                "sector": yf_data["info"].get("sector", ""),
                "industry": yf_data["info"].get("industry", ""),
                "track": track_name,
                "section": sections,
            })
            sources_used.append(f"yFinance ({ticker})")
            time.sleep(0.3)
        except Exception as e:
            print(f"yFinance error for {ticker}: {e}")
            nodes.append({"id": ticker, "name": ticker, "track": track_name})
    
    # --- Step 2: Find shared institutional holders ---
    print("\n  → Cross-holder analysis...")
    cross_holders = extract_shared_holders(tickers)
    for ch in cross_holders[:20]:
        # Shared holders = implicit relational link
        held = ch["holds"]
        for i in range(len(held)):
            for j in range(i+1, len(held)):
                links.append({
                    "pair": f"{held[i]} - {held[j]}",
                    "relationship": "Shared Institutional Holder",
                    "summary": f"{ch['institution']} holds stakes in both {held[i]} and {held[j]}.",
                    "sources": [f"yFinance institutional holders"],
                    "track": track_name,
                })
    
    # --- Step 3: SEC EDGAR 10-K relationships ---
    print("\n  → SEC EDGAR 10-K filings...")
    for ticker in tickers:
        try:
            cik = get_cik(ticker)
            filings = get_filings(cik, "10-K", count=1)
            if filings:
                f = filings[0]
                text = get_filing_text(cik, f["accessionNumber"], f["primaryDocument"])
                rels = extract_relationships_from_10k(text, ticker)
                
                for rel in rels:
                    # Check if target is one of our tracked companies
                    target_ticker = match_to_ticker(rel["target"], tickers)
                    if target_ticker:
                        links.append({
                            "pair": f"{ticker} - {target_ticker}",
                            "relationship": rel["type"].replace("_", " ").title(),
                            "summary": f"{ticker} references {rel['target']} as a {rel['type']} in their 10-K filing.",
                            "sources": [f"SEC EDGAR 10-K ({f['filingDate']})"],
                            "track": track_name,
                        })
                
                sources_used.append(f"SEC 10-K {ticker} ({f['filingDate']})")
                time.sleep(1)  # Respect SEC rate limits
        
        except Exception as e:
            print(f"    ⚠ SEC error for {ticker}: {e}")
    
    # --- Step 4: 13D/13G ownership links ---
    print("\n  → 13D/13G ownership filings...")
    for ticker in tickers:
        try:
            cik = get_cik(ticker)
            ownership = get_ownership_filings(cik)
            for o in ownership[:3]:
                links.append({
                    "pair": f"INVESTOR - {ticker}",
                    "relationship": "13D Activist Stake" if "13D" in o["form"] else "13G Passive Stake",
                    "summary": f"Investor filed {o['form']} for {ticker} on {o['date']}.",
                    "sources": [f"SEC EDGAR {o['form']} ({o['date']})"],
                    "track": track_name,
                })
            time.sleep(0.5)
        except Exception as e:
            print(f"    ⚠ 13D error for {ticker}: {e}")
    
    # Deduplicate links
    seen_pairs = set()
    unique_links = []
    for link in links:
        key = frozenset(link["pair"].split(" - "))
        if key not in seen_pairs:
            seen_pairs.add(key)
            unique_links.append(link)
    
    return {
        "track_name": track_name,
        "last_updated": datetime.now().isoformat(),
        "nodes": nodes,
        "links": unique_links,
        "metadata": {
            "total_nodes": len(nodes),
            "total_links": len(unique_links),
            "sources": list(set(sources_used)),
        }
    }

def match_to_ticker(company_name: str, tickers: list) -> str | None:
    """Fuzzy-match a company name string to a known ticker."""
    name_lower = company_name.lower()
    
    KNOWN_NAMES = {
        "salesforce": "CRM",
        "oracle": "ORCL",
        "servicenow": "NOW",
        "adobe": "ADBE",
        "workday": "WDAY",
    }
    
    for keyword, ticker in KNOWN_NAMES.items():
        if keyword in name_lower and ticker in tickers:
            return ticker
    
    return None

In [17]:
def save_to_txt(graph_data: dict, output_path: str = "relationships.txt"):
    """
    Save the relationship graph to a .txt file.
    Contains JSON data with a human-readable header.
    """
    with open(output_path, "w", encoding="utf-8") as f:
        # Human-readable header
        f.write("=" * 70 + "\n")
        f.write("COMPANY RELATIONSHIP GRAPH\n")
        f.write(f"Investment Track: {graph_data['track_name']}\n")
        f.write(f"Generated: {graph_data['last_updated']}\n")
        f.write(f"Nodes: {graph_data['metadata']['total_nodes']} | Links: {graph_data['metadata']['total_links']}\n")
        f.write("=" * 70 + "\n\n")
        
        # Quick summary table
        f.write("NODES:\n")
        for node in graph_data["nodes"]:
            f.write(f"  [{node['id']}] {node.get('name', node['id'])} — {node.get('industry', 'N/A')}\n")
        
        f.write("\nKEY RELATIONSHIPS:\n")
        for link in graph_data["links"][:10]:  # Top 10 preview
            f.write(f"  • {link['pair']}: {link['relationship']}\n")
            f.write(f"    {link['summary'][:120]}...\n")
        
        f.write("\n\n--- FULL JSON DATA ---\n\n")
        f.write(json.dumps(graph_data, indent=2))
    
    print(f"\n✅ Saved to: {output_path}")
    return output_path

def save_to_json(graph_data, output_path="relationships.json"):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(graph_data, f, indent=2)
    print(f"Saved to: {output_path}")


In [18]:
all_graphs = []
    
for track_name, tickers in INVESTMENT_TRACKS.items():
    graph = build_relationship_graph(tickers, track_name)
    all_graphs.append(graph)
    
    # Save individual track file
    save_to_txt(graph, f"relationships_{track_name.lower()}_sc.txt")
    save_to_json(graph, f"relationships_{track_name.lower()}_sc.json")

# Save combined master file
master = {
    "last_updated": datetime.now().isoformat(),
    "track_name": all_graphs,
    "nodes": [n for g in all_graphs for n in g["nodes"]],
    "links": [l for g in all_graphs for l in g["links"]],
    "metadata": {
        "total_nodes": sum(len(g["nodes"]) for g in all_graphs),
        "total_links": sum(len(g["links"]) for g in all_graphs),
    }
}

save_to_txt(master, "relationships_sc_MASTER.txt")
save_to_json(master, "relationships_sc_MASTER.json")


📊 Building graph for track: CLOUD_ENTERPRISE_SOFTWARE
   Tickers: CRM, ORCL, NOW, ADBE, WDAY

  → yFinance: CRM
  → yFinance: ORCL
  → yFinance: NOW
  → yFinance: ADBE
  → yFinance: WDAY

  → Cross-holder analysis...

  → SEC EDGAR 10-K filings...

  → 13D/13G ownership filings...

✅ Saved to: relationships_cloud_enterprise_software_sc.txt
Saved to: relationships_cloud_enterprise_software_sc.json

✅ Saved to: relationships_sc_MASTER.txt
Saved to: relationships_sc_MASTER.json


In [19]:
%pip install anthropic

Note: you may need to restart the kernel to use updated packages.


In [20]:
import anthropic
import os

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

SUPPLIER_EXTRACTION_PROMPT = """You are a financial document parser. Your only job is to extract DIRECT SUPPLIERS and MANUFACTURING PARTNERS from the provided SEC filing text.

STRICT RULES:
1. Only extract entities explicitly named as a supplier, vendor, or manufacturing partner of the filing company.
2. Do NOT extract competitors, customers, investors, or regulators.
3. Do NOT infer or assume relationships not explicitly stated in the text.
4. Do NOT hallucinate company names. Only use names that appear verbatim in the text.
5. If no qualifying relationships are found, return an empty suppliers list — do not guess.
6. Ignore vague mentions like "certain suppliers" with no named entity.
7. Return supplier names as their stock ticker symbol if you know it, otherwise use the exact company name from the text.

RELATIONSHIP TYPES YOU MAY EXTRACT:
- direct_supplier
- manufacturing_partner
- sole_source_supplier
- raw_material_vendor

RELATIONSHIP TYPES YOU MUST IGNORE:
- competitors, customers, investors, regulators, auditors, consultants

Return ONLY a JSON object in this exact format, no preamble, no explanation, no markdown backticks:
{
  "ticker": "<filing company ticker>",
  "suppliers": ["<ticker or name>", "<ticker or name>"]
}"""

In [21]:
def load_scraped_chunks(file_path) -> dict:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    if file_path.endswith(".json"):
        return json.loads(content)
    
    elif file_path.endswith(".txt"):
        marker = "--- FULL JSON DATA ---"
        if marker not in content:
            raise ValueError(f"No JSON marker found in {file_path}")
        json_str = content.split(marker)[-1].strip()
        return json.loads(json_str)
    
    else:
        raise ValueError(f"Unsupported file type: {file_path}")


def extract_suppliers_with_llm(
    section_text: str,
    company: str,
    section_name: str,
    model: str = "claude-haiku-4-5-20251001"
) -> dict:
    MAX_CHARS = 4000
    chunks = [section_text[i:i+MAX_CHARS] for i in range(0, len(section_text), MAX_CHARS)]
    
    all_suppliers = []
    
    for i, chunk in enumerate(chunks):
        print(f"  → LLM extraction: {company} / {section_name} / chunk {i+1} of {len(chunks)}")
        
        message = client.messages.create(
            model=model,
            max_tokens=1000,
            messages=[
                {
                    "role": "user",
                    "content": f"{SUPPLIER_EXTRACTION_PROMPT}\n\nFILING COMPANY TICKER: {company}\nSECTION: {section_name}\n\nTEXT:\n{chunk}"
                }
            ]
        )
        
        raw = next(block.text for block in message.content if block.type == "text").strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        
        try:
            parsed = json.loads(raw)
            suppliers = parsed.get("suppliers", [])
            
            # Filter out any "NONE" the LLM may have added itself before we aggregate
            suppliers = [s for s in suppliers if s != "NONE"]
            all_suppliers.extend(suppliers)
            
        except json.JSONDecodeError:
            print(f"    ⚠ JSON parse failed for chunk {i+1}, skipping")
            continue
    
    # Deduplicate
    seen = set()
    unique_suppliers = [s for s in all_suppliers if not (s in seen or seen.add(s))]
    
    # If nothing found across all chunks, return ["NONE"]
    if not unique_suppliers:
        unique_suppliers = ["NONE"]
    
    return {
        "ticker": company,
        "suppliers": unique_suppliers,
    }


def run_llm_extraction_from_file(file_path: str) -> list[dict]:
    """
    Master function — takes a scraped .txt or .json file,
    runs LLM extraction on each node's section text,
    and returns a list of {ticker, suppliers} dicts.
    """
    data = load_scraped_chunks(file_path)
    results = []
    
    # Each node in the scraped file corresponds to one company
    for node in data.get("nodes", []):
        company = node["id"]
        print(f"\n Processing: {company}")
        
        # Each node may have section texts attached from the scraper
        sections = node.get("sections", {})
        
        if not sections:
            print(f"No section text found for {company}, skipping")
            results.append({"ticker": company, "suppliers": ["NONE"]})
            continue
        
        company_suppliers = []
        
        for section_name, section_text in sections.items():
            if not section_text:
                continue
            
            llm_result = extract_suppliers_with_llm(section_text, company, section_name)
            
            # Only extend if real results found
            if llm_result["suppliers"] != ["NONE"]:
                company_suppliers.extend(llm_result["suppliers"])
        
        # Deduplicate across all sections for this company
        seen = set()
        unique = [s for s in company_suppliers if not (s in seen or seen.add(s))]
        
        results.append({
            "ticker": company,
            "suppliers": unique if unique else ["NONE"]
        })
    
    return results    

In [22]:
def debug_scraped_file(file_path: str):
    data = load_scraped_chunks(file_path)
    
    print("Top-level keys:", list(data.keys()))
    print("Number of nodes:", len(data.get("nodes", [])))
    
    for node in data.get("nodes", []):
        print(f"\n  Node keys for {node['id']}: {list(node.keys())}")
        print(f"  Has sections: {'sections' in node}")
        if "sections" in node:
            for sec_name, sec_text in node["sections"].items():
                print(f"    {sec_name}: {len(sec_text)} chars")

debug_scraped_file("relationships_cloud_enterprise_software_sc.json")

Top-level keys: ['track_name', 'last_updated', 'nodes', 'links', 'metadata']
Number of nodes: 5

  Node keys for CRM: ['id', 'name', 'sector', 'industry', 'track']
  Has sections: False

  Node keys for ORCL: ['id', 'name', 'sector', 'industry', 'track']
  Has sections: False

  Node keys for NOW: ['id', 'name', 'sector', 'industry', 'track']
  Has sections: False

  Node keys for ADBE: ['id', 'name', 'sector', 'industry', 'track']
  Has sections: False

  Node keys for WDAY: ['id', 'name', 'sector', 'industry', 'track']
  Has sections: False


In [107]:
input_file = "relationships_cloud_enterprise_software_sc.json"  # or .txt
    
supplier_results = run_llm_extraction_from_file(input_file)

# Save output
output = {"results": supplier_results}
with open("suppliers.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print("\n Done. Output:")
print(json.dumps(output, indent=2))


 Processing: CRM
No section text found for CRM, skipping

 Processing: ORCL
No section text found for ORCL, skipping

 Processing: NOW
No section text found for NOW, skipping

 Processing: ADBE
No section text found for ADBE, skipping

 Processing: WDAY
No section text found for WDAY, skipping

 Done. Output:
{
  "results": [
    {
      "ticker": "CRM",
      "suppliers": [
        "NONE"
      ]
    },
    {
      "ticker": "ORCL",
      "suppliers": [
        "NONE"
      ]
    },
    {
      "ticker": "NOW",
      "suppliers": [
        "NONE"
      ]
    },
    {
      "ticker": "ADBE",
      "suppliers": [
        "NONE"
      ]
    },
    {
      "ticker": "WDAY",
      "suppliers": [
        "NONE"
      ]
    }
  ]
}
